In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import time
from io import StringIO

# 크롬 드라이버 경로 설정
driver_path = r"C:\chromedriver-win64\chromedriver-win64\chromedriver.exe"

options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(driver_path), options=options)

# 접속
url = "https://stat.molit.go.kr/portal/cate/statView.do?hRsId=58&hFormId=5498&hSelectId=1244&hPoint=00&hAppr=1&hDivEng=&oFileName=&rFileName=&midpath=&sFormId=5498&sStart=2024&sEnd=2024&sStyleNum=562&settingRadio=xlsx"
driver.get(url)
time.sleep(5)

# ✅ 테이블이 들어 있는 스크롤 div 찾아서 끝까지 내리기
scrollable_div = driver.find_element(By.CLASS_NAME, "LayoutTable" "")

# 강제로 스크롤 내려서 모든 데이터 로딩
for _ in range(30):
    driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scrollable_div)
    time.sleep(0.2)


# ✅ 테이블 HTML 가져오기
table_html = driver.find_element(By.ID, "sheet01-table").get_attribute("outerHTML")
driver.quit()

# ✅ BeautifulSoup으로 파싱
soup = BeautifulSoup(table_html, "html.parser")
table = soup.find("table")

# ✅ 테이블 파싱
rows = table.find_all("tr")
data = []
for row in rows:
    cols = row.find_all(["td", "th"])
    cols_text = [col.get_text(strip=True) for col in cols]
    data.append(cols_text)

# ✅ pandas DataFrame 생성
df = pd.DataFrame(data)
df.columns = df.iloc[0]
df = df.drop(index=0).reset_index(drop=True)

# ✅ 숫자형 변환 (쉼표 제거)
for col in df.columns[1:]:
    df[col] = df[col].astype(str).str.replace(",", "").str.strip()
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 결과 출력
print(df.head())


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":".LayoutTable"}
  (Session info: chrome=135.0.7049.42); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x00007FF7E05C1F55+78133]
	GetHandleVerifier [0x00007FF7E05C1FB0+78224]
	(No symbol) [0x00007FF7E03891BA]
	(No symbol) [0x00007FF7E03DF19D]
	(No symbol) [0x00007FF7E03DF44C]
	(No symbol) [0x00007FF7E04323D7]
	(No symbol) [0x00007FF7E040719F]
	(No symbol) [0x00007FF7E042F21F]
	(No symbol) [0x00007FF7E0406F33]
	(No symbol) [0x00007FF7E03D0358]
	(No symbol) [0x00007FF7E03D10C3]
	GetHandleVerifier [0x00007FF7E088BA8D+3001453]
	GetHandleVerifier [0x00007FF7E0885E72+2977874]
	GetHandleVerifier [0x00007FF7E08A497D+3103581]
	GetHandleVerifier [0x00007FF7E05DC7EA+186826]
	GetHandleVerifier [0x00007FF7E05E43FF+218591]
	GetHandleVerifier [0x00007FF7E05C9D94+